# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [2]:
!pip install beir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 15.1 MB/s eta 0:00:00


In [3]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [4]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

'../data/beir_datasets/scifact'

In [5]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [6]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [7]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [8]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [9]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


In [10]:
!pip install rank_bm25 sentence-transformers xgboost beir

## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [11]:
import numpy as np
from rank_bm25 import BM25Okapi
from beir.retrieval.evaluation import EvaluateRetrieval
import string

In [14]:
tokens_corpus = [x.split(" ") for x in df_corpus['text'].tolist()]

bm25_model = BM25Okapi(tokens_corpus)

def retrieve_bm25(query_text, top_k=100):
    query_tokens = query_text.split(" ")
    score_list = bm25_model.get_scores(query_tokens)
    ranked_idx = np.argsort(score_list)[::-1][:top_k]

    hits = {}
    for i in ranked_idx:
        doc_id = df_corpus.iloc[i]['doc_id']
        hits[doc_id] = float(score_list[i])
    return hits

In [17]:
bm25_outputs = {}
for _, q_row in df_queries.iterrows():
    q_id = q_row['query_id']
    q_text = q_row['query']
    bm25_outputs[q_id] = retrieve_bm25(q_text, top_k=100)

print("BM25 Baseline")
ndcg, avg_map, recall, _ = EvaluateRetrieval.evaluate(qrels, bm25_outputs, k_values=[10])
print(f"nDCG@10 = {ndcg['NDCG@10']:.4f}")
print(f"Recall@10 = {recall['Recall@10']:.4f}")

BM25 Baseline
nDCG@10 = 0.5056
Recall@10 = 0.6247


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [18]:
from sentence_transformers import CrossEncoder

ce_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

ce_outputs = {}

for q_id, bm25_hits in bm25_outputs.items():
    q_text = df_queries[df_queries['query_id'] == q_id]['query'].values[0]

    pair_list = []
    doc_keys = []

    for d_id in bm25_hits:
        d_text = df_corpus[df_corpus['doc_id'] == d_id]['text'].values[0]
        pair_list.append([q_text, d_text])
        doc_keys.append(d_id)

    if len(pair_list) > 0:
        ce_scores = ce_model.predict(pair_list)

        ranked_hits = {}
        for i, d_id in enumerate(doc_keys):
            ranked_hits[d_id] = float(ce_scores[i])

        ce_outputs[q_id] = dict(
            sorted(ranked_hits.items(), key=lambda x: x[1], reverse=True)
        )

2026-01-13 16:29:20.135619: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768321760.318236      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768321760.368392      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768321760.786438      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768321760.786478      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768321760.786481      55 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [26]:
test_qid = "100"
print(f"Resultados de re-ranking para la consulta {test_qid}")

bm25_top10 = list(bm25_outputs[test_qid].keys())[:10]
ce_top10 = list(ce_outputs[test_qid].keys())[:10]

print("Top 10 con BM25:", bm25_top10)
print("Top 10 después del Cross-Encoder:", ce_top10)

docs_nuevos = set(ce_top10) - set(bm25_top10)
print("Documentos que ingresan al Top 10 tras el re-ranking:", docs_nuevos)

Resultados de re-ranking para la consulta 100
Top 10 con BM25: ['4381486', '2547636', '22186938', '34982259', '7583161', '40234452', '20186814', '3391547', '2701077', '10526279']
Top 10 después del Cross-Encoder: ['4381486', '14550841', '2701077', '40234452', '25516011', '92499', '30714190', '17682477', '1944452', '9911547']
Documentos que ingresan al Top 10 tras el re-ranking: {'14550841', '25516011', '9911547', '17682477', '92499', '30714190', '1944452'}


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [27]:
import xgboost as xgb

features = []
labels = []
group_sizes = []
pair_index = []

for q_id in bm25_outputs:
    if q_id in qrels:
        bm25_hits = bm25_outputs[q_id]
        rel_docs = qrels[q_id]

        docs_in_query = 0
        for d_id, bm25_score in bm25_hits.items():
            doc_text = df_corpus[df_corpus['doc_id'] == d_id]['text'].values[0]
            doc_length = len(doc_text.split())

            features.append([bm25_score, doc_length])
            labels.append(rel_docs.get(d_id, 0))

            pair_index.append((q_id, d_id))
            docs_in_query += 1

        group_sizes.append(docs_in_query)

X = np.array(features)
y = np.array(labels)

In [28]:
ltr_model = xgb.XGBRanker(
    tree_method="hist",
    objective="rank:ndcg",
    eval_metric="ndcg@10"
)

ltr_model.fit(X, y, group=group_sizes)

XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=None, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric='ndcg@10', feature_types=None, feature_weights=None,
          gamma=None, grow_policy=None, importance_type=None,
          interaction_constraints=None, learning_rate=None, max_bin=None,
          max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=None,
          max_depth=None, max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=None,
          n_jobs=None, num_parallel_tree=None, ...)

In [29]:
ltr_scores = ltr_model.predict(X)

In [32]:
ltr_outputs = {}

for i, (q_id, d_id) in enumerate(pair_index):
    if q_id not in ltr_outputs:
        ltr_outputs[q_id] = {}
    ltr_outputs[q_id][d_id] = float(ltr_scores[i])

print("Modelo LTR entrenado y resultados generados correctamente.")

Modelo LTR entrenado y resultados generados correctamente.


In [33]:
sample_qid = "100"
top_docs_ltr = sorted(
    ltr_outputs[sample_qid].items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print(f"Top documentos para la consulta {sample_qid} usando LTR:")
print([doc_id for doc_id, _ in top_docs_ltr[:5]])

Top documentos para la consulta 100 usando LTR:
['4381486', '2547636', '2701077', '22186938', '10526279']


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [34]:
methods_eval = {
    "BM25": bm25_outputs,
    "Cross-Encoder": ce_outputs,
    "LTR (XGBoost)": ltr_outputs
}

metrics_rows = []

In [35]:
print("Comparación final de modelos")

for method_name, method_results in methods_eval.items():
    ndcg, avg_map, recall, _ = EvaluateRetrieval.evaluate(
        qrels,
        method_results,
        k_values=[10]
    )

    metrics_rows.append({
        "Modelo": method_name,
        "nDCG@10": ndcg["NDCG@10"],
        "MAP@10": avg_map["MAP@10"],
        "Recall@10": recall["Recall@10"]
    })

Comparación final de modelos


In [36]:
df_metrics = pd.DataFrame(metrics_rows)
df_metrics

,Modelo,nDCG@10,MAP@10,Recall@10
0,BM25,0.50562,0.46273,0.62472
1,Cross-Encoder,0.62616,0.58890,0.72350
2,LTR (XGBoost),0.74118,0.72927,0.75733
